In [3]:
import zipfile

zip_path = "/content/MedQuAD-master.zip"
extract_path = "MedQuAD"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted Successfully")

Extracted Successfully


In [5]:
import os

os.listdir("MedQuAD")

['MedQuAD-master']

In [7]:
import os
import xml.etree.ElementTree as ET

data = []

base_path = "MedQuAD"

for root_dir, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".xml"):
            file_path = os.path.join(root_dir, file)

            tree = ET.parse(file_path)
            root = tree.getroot()

            # 🔥 IMPORTANT CHANGE
            for qa in root.iter("QAPair"):

                question = qa.find("Question")
                answer = qa.find("Answer")

                if question is not None and answer is not None:
                    if question.text and answer.text:
                        data.append((question.text.strip(), answer.text.strip()))

print("Total QA Pairs:", len(data))

Total QA Pairs: 16407


In [8]:
questions = [q for q, a in data]
answers = [a for q, a in data]

print(len(questions))

16407


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(questions)

print("Vectorization Done")

Vectorization Done


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

def get_answer(query):
    query_vec = vectorizer.transform([query])

    similarity = cosine_similarity(query_vec, X)

    index = similarity.argmax()

    return answers[index]

In [11]:
print(get_answer("What are symptoms of diabetes?"))

Diabetes is often called a "silent" disease because it can cause serious complications even before you have symptoms. Symptoms can also be so mild that you dont notice them. An estimated 8 million people in the United States have type 2 diabetes and dont know it, according to 2012 estimates by the Centers for Disease Control and Prevention (CDC). Common Signs Some common symptoms of diabetes are: - being very thirsty  - frequent urination  - feeling very hungry or tired  - losing weight without trying  - having sores that heal slowly  - having dry, itchy skin  - loss of feeling or tingling in the feet  - having blurry eyesight. being very thirsty frequent urination feeling very hungry or tired losing weight without trying having sores that heal slowly having dry, itchy skin loss of feeling or tingling in the feet having blurry eyesight. Signs of type 1 diabetes usually develop over a short period of time. The signs for type 2 diabetes develop more gradually. Tests for Diabetes The foll

In [12]:
def extract_entities(text):

    diseases = ["diabetes", "cancer", "asthma", "covid", "fever"]
    symptoms = ["pain", "cough", "headache", "fatigue"]

    found = []

    for word in diseases + symptoms:
        if word in text.lower():
            found.append(word)

    return list(set(found))

In [13]:
query = "What are symptoms of asthma?"

print("Answer:", get_answer(query))
print("Entities:", extract_entities(query))

Answer: What are the signs and symptoms of Asthma? The Human Phenotype Ontology provides the following list of signs and symptoms for Asthma. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate number of patients (when available) Asthma - The Human Phenotype Ontology (HPO) has collected information on how often a sign or symptom occurs in a condition. Much of this information comes from Orphanet, a European rare disease database. The frequency of a sign or symptom is usually listed as a rough estimate of the percentage of patients who have that feature. The frequency may also be listed as a fraction. The first number of the fraction is how many people had the symptom, and the second number is the total number of people who were examined in one study. For example, a frequency of 25/25 means th